In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Ruta a la carpeta del proyecto (una carpeta "arriba" de notebooks/)
RUTA = ".."

# Cargamos el CSV que guardamos en el notebook 1
df = pd.read_csv(f"{RUTA}/data/raw/demanda_ree.csv")

print(df.shape)
df.info()


(296871, 2)
<class 'pandas.DataFrame'>
RangeIndex: 296871 entries, 0 to 296870
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   datetime    296871 non-null  str  
 1   demanda_mw  296871 non-null  int64
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [2]:
# Convertimos el texto en fecha (en UTC, para evitar problemas con el cambio de hora)
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 296871 entries, 0 to 296870
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype              
---  ------      --------------   -----              
 0   datetime    296871 non-null  datetime64[us, UTC]
 1   demanda_mw  296871 non-null  int64              
dtypes: datetime64[us, UTC](1), int64(1)
memory usage: 4.5 MB


In [3]:
# 1. Ponemos la fecha como índice (resample lo necesita)
df = df.set_index("datetime")

# 2. Agrupamos por horas y calculamos la media de cada hora
df_horario = df.resample("h").mean()

print(df_horario.shape)
print(df_horario.head())


(24783, 1)
                             demanda_mw
datetime                               
2023-12-31 23:00:00+00:00  23352.166667
2024-01-01 00:00:00+00:00  22610.750000
2024-01-01 01:00:00+00:00  21386.083333
2024-01-01 02:00:00+00:00  20141.333333
2024-01-01 03:00:00+00:00  19354.583333


In [4]:
# Ahora a encontrar los huecos
df_horario[df_horario["demanda_mw"].isna()]

#Hay dos tipos de huecos, y cada uno pide una solución distinta.

#Huecos cortos (1–2 horas): si interpolar
#La demanda no cambia de golpe en 1 o 2 horas. Unir con una línea el dato anterior y el siguiente da un valor muy realista.
#Huecos largos (8–17 horas): no interpolar


,demanda_mw
datetime,
2024-01-07 03:00:00+00:00,NaN
2024-01-07 04:00:00+00:00,NaN
2024-01-07 05:00:00+00:00,NaN
2024-01-07 06:00:00+00:00,NaN
2024-01-07 07:00:00+00:00,NaN
2024-01-07 08:00:00+00:00,NaN
2024-01-07 09:00:00+00:00,NaN
2024-01-07 10:00:00+00:00,NaN
2024-01-07 11:00:00+00:00,NaN


In [5]:
# Rellenamos

# Guardamos cuántos NaN hay antes, para comprobar
print("NaN antes:", df_horario["demanda_mw"].isna().sum())

# PASO 1: huecos cortos =  interpolar, pero como máximo 2 horas seguidas
df_horario["demanda_mw"] = df_horario["demanda_mw"].interpolate(limit=2)
print("NaN tras interpolar:", df_horario["demanda_mw"].isna().sum())

# PASO 2: huecos largos = copiar el valor de hace 1 semana (168 horas)
semana_anterior = df_horario["demanda_mw"].shift(168)
df_horario["demanda_mw"] = df_horario["demanda_mw"].fillna(semana_anterior)
print("NaN final:", df_horario["demanda_mw"].isna().sum())


NaN antes: 40
NaN tras interpolar: 29
NaN final: 15


Quedan exactamente 15. ¿casualidad? la tabla: son las 15 horas que quedaban del 7 de enero de 2024. Los huecos de junio sí se han rellenado.

¿Por que el 7 de enero no?: shift(168) copia el valor de hace una semana.
Una semana antes del 7 de enero de 2024 es el 31 de diciembre de 2023.

¿Desde qué fecha descargué los datos?
Los datos empiezan el 1 de enero de 2024. El 31 de diciembre de 2023 no está en el dataset, así que shift(168) devuelve NaN y no hay nada con qué rellenar.


In [6]:
# si no hay semana anterior, usamos la semana siguiente (-168)
semana_siguiente = df_horario["demanda_mw"].shift(-168)
df_horario["demanda_mw"] = df_horario["demanda_mw"].fillna(semana_siguiente)
print("NaN final:", df_horario["demanda_mw"].isna().sum())
#"KeyError tras set_index: la columna pasa a ser el índice. Solución: ejecutar el notebook en orden de principio a fin."



NaN final: 0


In [7]:
# A: pasar a hora de Madrid
# Convertimos el índice de UTC a hora de Madrid
df_horario.index = df_horario.index.tz_convert("Europe/Madrid")

print(df_horario.head())


                             demanda_mw
datetime                               
2024-01-01 00:00:00+01:00  23352.166667
2024-01-01 01:00:00+01:00  22610.750000
2024-01-01 02:00:00+01:00  21386.083333
2024-01-01 03:00:00+01:00  20141.333333
2024-01-01 04:00:00+01:00  19354.583333


In [8]:
df_horario.to_csv(f"{RUTA}/data/processed/demanda_horaria.csv")
print("Guardado ✅")


Guardado ✅


# TEMPERATURA

In [9]:
import requests

url_meteo = "https://archive-api.open-meteo.com/v1/archive"
params_meteo = {
    "latitude": 40.4168,       # Madrid
    "longitude": -3.7038,
    "start_date": "2023-12-31",
    "end_date": "2026-10-29",
    "hourly": "temperature_2m",   # temperatura a 2 metros del suelo
    "timezone": "UTC"             # en UTC, igual que hicimos con la demanda
}

r = requests.get(url_meteo, params=params_meteo)
print(r.status_code)
meteo = r.json()


200


In [10]:
type(meteo["latitude"])
print(type(meteo))
print(meteo.keys())
#busco hourly y temperature
# Busco los datos horarios
print(type(meteo["hourly"]))
print(meteo["hourly"].keys())
# compruebo con len y [0]
# Las horas
print(len(meteo["hourly"]["time"]))
print(meteo["hourly"]["time"][0])

# Las temperaturas
print(len(meteo["hourly"]["temperature_2m"]))
print(meteo["hourly"]["temperature_2m"][0])



<class 'dict'>
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
<class 'dict'>
dict_keys(['time', 'temperature_2m'])
24816
2023-12-31T00:00
24816
3.9


In [11]:
# Indice con df_temp
df_temp = pd.DataFrame({
    "datetime": meteo["hourly"]["time"],                  # la lista de horas
    "temperatura": meteo["hourly"]["temperature_2m"]      # la lista de temperaturas
})

print(df_temp.head())
print(df_temp.shape)


           datetime  temperatura
0  2023-12-31T00:00          3.9
1  2023-12-31T01:00          5.6
2  2023-12-31T02:00          5.5
3  2023-12-31T03:00          5.6
4  2023-12-31T04:00          6.4
(24816, 2)


In [12]:
# Convertir ese indice a hora de Madrid
df_temp["datetime"] = pd.to_datetime(df_temp["datetime"], utc=True)

print(df_temp.head())
print(df_temp.shape)
df_temp.info()


                   datetime  temperatura
0 2023-12-31 00:00:00+00:00          3.9
1 2023-12-31 01:00:00+00:00          5.6
2 2023-12-31 02:00:00+00:00          5.5
3 2023-12-31 03:00:00+00:00          5.6
4 2023-12-31 04:00:00+00:00          6.4
(24816, 2)
<class 'pandas.DataFrame'>
RangeIndex: 24816 entries, 0 to 24815
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   datetime     24816 non-null  datetime64[us, UTC]
 1   temperatura  24816 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(1)
memory usage: 387.9 KB


In [13]:
# Unimos: a cada hora de demanda le pegamos su temperatura
df_final = df_horario.join(df_temp, how="left")

print(df_final.head())
print(df_final.shape)
print(df_final.isna().sum())


                             demanda_mw datetime  temperatura
datetime                                                     
2024-01-01 00:00:00+01:00  23352.166667      NaT          NaN
2024-01-01 01:00:00+01:00  22610.750000      NaT          NaN
2024-01-01 02:00:00+01:00  21386.083333      NaT          NaN
2024-01-01 03:00:00+01:00  20141.333333      NaT          NaN
2024-01-01 04:00:00+01:00  19354.583333      NaT          NaN
(24783, 3)
demanda_mw         0
datetime       24783
temperatura    24783
dtype: int64


In [14]:
# Paso 1: la fecha pasa a ser el índice (¡y lo GUARDAMOS!)
df_temp = df_temp.set_index("datetime")

# Paso 2: convertimos el índice a hora de Madrid
df_temp.index = df_temp.index.tz_convert("Europe/Madrid")

# Comprobación: ahora debería tener solo 1 columna (temperatura)
print(df_temp.head())


#join devolvía todo NaN: el set_index no se había guardado y los índices (fechas frente a números) no coincidían. Lección: comprobar siempre el resultado de un join con .isna().sum()

                           temperatura
datetime                              
2023-12-31 01:00:00+01:00          3.9
2023-12-31 02:00:00+01:00          5.6
2023-12-31 03:00:00+01:00          5.5
2023-12-31 04:00:00+01:00          5.6
2023-12-31 05:00:00+01:00          6.4


In [15]:
#El join sirve para juntar las dos tablas en una sola
df_final = df_horario.join(df_temp, how="left")

print(df_final.head())
print(df_final.shape)
print(df_final.isna().sum())


                             demanda_mw  temperatura
datetime                                            
2024-01-01 00:00:00+01:00  23352.166667          4.6
2024-01-01 01:00:00+01:00  22610.750000          4.3
2024-01-01 02:00:00+01:00  21386.083333          4.1
2024-01-01 03:00:00+01:00  20141.333333          3.6
2024-01-01 04:00:00+01:00  19354.583333          3.1
(24783, 2)
demanda_mw     0
temperatura    0
dtype: int64


In [16]:
df_final.to_csv(f"{RUTA}/data/processed/dataset_final.csv")
print("Guardado ")


Guardado 
